In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm #need to install statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import dataframe_image as dfi

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [3]:
df_assess = pd.read_csv("data/cleaned/final_assessment.csv")
df_grades = pd.read_csv("data/cleaned/grades.csv")
df_status = pd.read_csv("data/cleaned/status.csv")
df_intake = pd.read_csv("data/cleaned/intake_form.csv")

In [4]:
df_intake

,id,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,DSC 180,major,neither,Third year transfer,MATH 183,2,2,4
9,24,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


## Merge two dataframes(grades and status) by id, create coding and handwritten binary columns for b1 vs b2 test. I got rid of section 4 for now since they will affect both b1(coefficient of coding) and b2(coefficient of handwritten).

In [5]:
df_status = df_status[df_status['completed'] == 1]

sample = df_grades.merge(df_status[["id", "section"]], on="id", how="inner")

sample['coding'] =  ((sample['section'] == 2) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
sample['handwritten'] =  ((sample['section'] == 3) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
#sample_filter = sample[(sample['coding'] == 0) | (sample['handwritten'] == 0)]
sample

,id,coding_score,handwritten_score,final_score,final_score_adj,section,coding,handwritten
0,0,NaN,NaN,0.737705,0.586066,1,0,0
1,1,0.928571,NaN,0.754098,0.610656,2,1,0
2,3,0.857143,1.0,0.836066,0.479508,4,1,1
3,6,NaN,1.0,0.459016,0.200820,3,0,1
4,9,0.904764,NaN,0.672131,0.471311,2,1,0
5,11,0.928571,1.0,0.918033,0.319672,4,1,1
6,12,NaN,NaN,0.098361,0.000000,1,0,0
7,15,1.000000,1.0,0.344262,0.159836,4,1,1
8,20,NaN,NaN,0.459016,0.274590,1,0,0
9,24,NaN,NaN,1.000000,0.754098,1,0,0


## Linear model for final_score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [6]:
res = smf.ols(
    formula='final_score ~ coding * handwritten',
    data=sample
).fit()

print(res.summary())

s1 = res.summary2()
df_model = s1.tables[0]
df_coef  = s1.tables[1]
dfi.export(df_coef,  "data/statistic_analysis_outputs/final_separate.png", table_conversion="chrome", dpi = 300)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.041
Model:                            OLS   Adj. R-squared:                 -0.181
Method:                 Least Squares   F-statistic:                    0.1837
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.906
Time:                        14:20:11   Log-Likelihood:                 2.2359
No. Observations:                  17   AIC:                             3.528
Df Residuals:                      13   BIC:                             6.861
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.5705      0

## Linear model is final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [7]:
res2 = smf.ols(
    formula='final_score ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res2.summary())
#anova_results = anova_lm(model2, model1)
s2 = res2.summary2()
df_model2 = s2.tables[0]
df_coef2  = s2.tables[1]
dfi.export(df_coef2,  "data/statistic_analysis_outputs/final_combine.png", table_conversion="chrome", dpi = 300)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                 -0.097
Method:                 Least Squares   F-statistic:                    0.2917
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.751
Time:                        14:20:13   Log-Likelihood:                 2.2299
No. Observations:                  17   AIC:                             1.540
Df Residuals:                      14   BIC:                             4.040
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models

In [8]:
anova_results = sm.stats.anova_lm(res2, res)
print(anova_results)
dfi.export(
    anova_results,
    "data/statistic_analysis_outputs/final_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      14.0  0.765668      0.0       NaN       NaN       NaN
1      13.0  0.765130      1.0  0.000537  0.009132  0.925325


## Linear model for adjusted final score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [9]:
res3 = smf.ols(
    formula='final_score_adj ~ coding * handwritten',
    data=sample
).fit()

print(res3.summary())

s3 = res3.summary2()
df_model3 = s3.tables[0]
df_coef3  = s3.tables[1]
dfi.export(df_coef3,  "data/statistic_analysis_outputs/final_adj_separate.png", table_conversion="chrome", dpi = 300)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                 -0.141
Method:                 Least Squares   F-statistic:                    0.3390
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.798
Time:                        14:20:15   Log-Likelihood:                 6.0964
No. Observations:                  17   AIC:                            -4.193
Df Residuals:                      13   BIC:                           -0.8600
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3803      0

## Linear model is adjusted final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [10]:
res4 = smf.ols(
    formula='final_score_adj ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res4.summary())
#anova_results = anova_lm(model2, model1)
s4 = res4.summary2()
df_model4 = s4.tables[0]
df_coef4  = s4.tables[1]
dfi.export(df_coef4,  "data/statistic_analysis_outputs/final_adj_combine.png", table_conversion="chrome", dpi = 300)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                 -0.101
Method:                 Least Squares   F-statistic:                    0.2642
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.772
Time:                        14:20:17   Log-Likelihood:                 5.7712
No. Observations:                  17   AIC:                            -5.542
Df Residuals:                      14   BIC:                            -3.043
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models for adjusted final score

In [11]:
anova_results2 = sm.stats.anova_lm(res4, res3)
print(anova_results2)
dfi.export(
    anova_results2,
    "data/statistic_analysis_outputs/final_adj_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff        F   Pr(>F)
0      14.0  0.504782      0.0       NaN      NaN      NaN
1      13.0  0.485833      1.0  0.018949  0.50703  0.48901


## Including variables in intake form

In [12]:
new_sample = sample.merge(df_intake, on="id", how="inner")
sample_new = new_sample.drop(columns = ['coding_score', 'handwritten_score', 'section'])
sample_new

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.586066,0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,0.754098,0.610656,1,0,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,0.836066,0.479508,1,1,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,0.459016,0.200820,0,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,0.672131,0.471311,1,0,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,0.918033,0.319672,1,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,0.098361,0.000000,0,0,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,0.344262,0.159836,1,1,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,0.459016,0.274590,0,0,DSC 180,major,neither,Third year transfer,MATH 183,2,2,4
9,24,1.000000,0.754098,0,0,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


In [13]:
sample_new['recruitment_source'].value_counts()
#If I remember it right, class_standing(student's year) is redundant with this feature
#since older students are likely to recruited from a uppertive class

heard_type = {
    'DSC 10': 'Heard_1',
    'DSC 20': 'Heard_1',
    'DSC 30': 'Heard_1',
    'DSC 40A': 'Heard_2',
    'DSC 40B': 'Heard_2',
    'DSC 80': 'Heard_2',
    'DSC 140B': 'Heard_3',
    'CSE 150A': 'Heard_3',
    'CSE 151A': 'Heard_3',
    'DSC 180': 'Heard_3',
    'DSC 100': 'Heard_4',
    'DSC 120': 'Heard_4',
    'DSC 190': 'Heard_4',
}

In [14]:
sample_new['dsc_affiliation'].value_counts()
dsc_type = {
    'major': 'DSC_major',
    'minor': 'DSC_minor',
    'neither': 'DSC_neither',
}

In [15]:
sample_new['math_affiliation'].value_counts()

math_type = {
    'major': 'MATH_major',
    'minor': 'MATH_minor',
    'neither': 'MATH_neither',
}

In [16]:
sample_new['class_standing'].value_counts()

class_standing
Third year                            6
Second year                           4
Fourth year (second year transfer)    2
Fourth year                           2
Third year (first year transfer)      1
Third year transfer                   1
third year transfer                   1
Name: count, dtype: int64

In [17]:
sample_new['stats_courses_taken'].value_counts()

section_type = {
    'MATH 181A': 'Stats_1',
    'MATH 180A': 'Stats_2',
    'ECE 109': 'Stats_2',
    'MAE 108': 'Stats_2',
    'MATH 183': 'Stats_3',
    'MATH 186': 'Stats_3',
    'ECON 120A': 'Stats_3',
    'None of the above': 'Stats_4'
}

In [18]:
def mappings(col, dictionary):
    values = [i.strip() for i in col.split(',')]
    adjust = {dictionary.get(j) for j in values if j in dictionary} #keep unique values
    return ', '.join(adjust)

In [19]:
samples = sample_new.copy()
samples['recruitment_source'] = samples['recruitment_source'].apply(lambda x: mappings(x, heard_type))
samples['stats_courses_taken'] = samples['stats_courses_taken'].apply(lambda x: mappings(x, section_type))
samples['dsc_affiliation'] = samples['dsc_affiliation'].apply(lambda x: mappings(x, dsc_type))
samples['math_affiliation'] = samples['math_affiliation'].apply(lambda x: mappings(x, math_type))
samples

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.586066,0,0,Heard_3,DSC_major,MATH_neither,Second year,"Stats_2, Stats_1",4,2,5
1,1,0.754098,0.610656,1,0,Heard_3,DSC_major,MATH_neither,Third year,Stats_2,4,3,5
2,3,0.836066,0.479508,1,1,Heard_3,DSC_major,MATH_neither,Third year,Stats_3,4,3,4
3,6,0.459016,0.200820,0,1,Heard_3,DSC_major,MATH_neither,Second year,Stats_3,3,1,4
4,9,0.672131,0.471311,1,0,Heard_3,DSC_major,MATH_major,Third year,Stats_3,4,2,4
5,11,0.918033,0.319672,1,1,Heard_3,DSC_major,MATH_neither,Second year,Stats_3,3,1,4
6,12,0.098361,0.000000,0,0,Heard_3,DSC_major,MATH_neither,Third year,Stats_3,4,4,4
7,15,0.344262,0.159836,1,1,Heard_2,DSC_major,MATH_neither,Third year (first year transfer),Stats_3,3,1,4
8,20,0.459016,0.274590,0,0,Heard_3,DSC_major,MATH_neither,Third year transfer,Stats_3,2,2,4
9,24,1.000000,0.754098,0,0,Heard_3,DSC_major,MATH_neither,Fourth year (second year transfer),"Stats_3, Stats_2, Stats_1",5,5,5


In [20]:
df_encoded = pd.concat([samples, samples['stats_courses_taken'].str.get_dummies(sep=', '), 
                        samples['recruitment_source'].str.get_dummies(sep=', '),
                        samples['dsc_affiliation'].str.get_dummies(sep=', '),
                        samples['math_affiliation'].str.get_dummies(sep=', '),], axis = 1)
df_encoded = df_encoded.drop(columns = ['recruitment_source', 'dsc_affiliation', 'math_affiliation', 'stats_courses_taken', 'Stats_4',
                           'DSC_neither', 'MATH_neither'], errors='ignore') 
#dropping all string columns and neither columns to prevent redundency
df_encoded

,id,final_score,final_score_adj,coding,handwritten,class_standing,stats_confidence,chebyshev_familiarity,python_skill_level,Stats_1,Stats_2,Stats_3,Heard_2,Heard_3,Heard_4,DSC_major,DSC_minor,MATH_major
0,0,0.737705,0.586066,0,0,Second year,4,2,5,1,1,0,0,1,0,1,0,0
1,1,0.754098,0.610656,1,0,Third year,4,3,5,0,1,0,0,1,0,1,0,0
2,3,0.836066,0.479508,1,1,Third year,4,3,4,0,0,1,0,1,0,1,0,0
3,6,0.459016,0.200820,0,1,Second year,3,1,4,0,0,1,0,1,0,1,0,0
4,9,0.672131,0.471311,1,0,Third year,4,2,4,0,0,1,0,1,0,1,0,1
5,11,0.918033,0.319672,1,1,Second year,3,1,4,0,0,1,0,1,0,1,0,0
6,12,0.098361,0.000000,0,0,Third year,4,4,4,0,0,1,0,1,0,1,0,0
7,15,0.344262,0.159836,1,1,Third year (first year transfer),3,1,4,0,0,1,1,0,0,1,0,0
8,20,0.459016,0.274590,0,0,Third year transfer,2,2,4,0,0,1,0,1,0,1,0,0
9,24,1.000000,0.754098,0,0,Fourth year (second year transfer),5,5,5,1,1,1,0,1,0,1,0,0


In [21]:
df_encoded.columns

Index(['id', 'final_score', 'final_score_adj', 'coding', 'handwritten', 'class_standing', 'stats_confidence', 'chebyshev_familiarity', 'python_skill_level',
       'Stats_1', 'Stats_2', 'Stats_3', 'Heard_2', 'Heard_3', 'Heard_4', 'DSC_major', 'DSC_minor', 'MATH_major'],
      dtype='object')

In [22]:
res5 = smf.ols( #no heard 1 stat, no MATH_minor in the sample
    formula='final_score ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_1 + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res5.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                 -1.895
Method:                 Least Squares   F-statistic:                    0.1943
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.985
Time:                        14:20:20   Log-Likelihood:                 7.0752
No. Observations:                  17   AIC:                             13.85
Df Residuals:                       3   BIC:                             25.51
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.44

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [23]:
coeffs = pd.DataFrame({
    'coef': res5.params,
    'std_err': res5.bse,
    't': res5.tvalues,
    'p': res5.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.445368  0.447464  0.995314  0.392945
coding                 0.052186  0.334747  0.155896  0.886014
handwritten           -0.132927  0.443578 -0.299669  0.783992
coding:handwritten     0.156098  0.554106  0.281711  0.796480
stats_confidence       0.166862  0.251518  0.663420  0.554487
chebyshev_familiarity -0.110955  0.145033 -0.765027  0.499930
python_skill_level    -0.064398  0.210574 -0.305821  0.779734
Stats_1                0.030494  0.585721  0.052062  0.961752
Stats_2                0.220511  0.734155  0.300361  0.783513
Stats_3                0.084782  0.439229  0.193024  0.859269
Heard_2               -0.144676  0.289626 -0.499526  0.651745
Heard_3                0.034188  0.227628  0.150194  0.890141
Heard_4                0.555855  0.513534  1.082411  0.358309
DSC_major             -0.010749  0.278247 -0.038630  0.971613
DSC_minor              0.456116  0.483730  0.942915  0.415287
MATH_maj

## Drop DSC_major 

In [24]:
res5 = smf.ols( #Drop DSC_major 
    formula='final_score ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_1 + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res5.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                 -1.895
Method:                 Least Squares   F-statistic:                    0.1943
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.985
Time:                        14:20:20   Log-Likelihood:                 7.0752
No. Observations:                  17   AIC:                             13.85
Df Residuals:                       3   BIC:                             25.51
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.43

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [25]:
coeffs = pd.DataFrame({
    'coef': res5.params,
    'std_err': res5.bse,
    't': res5.tvalues,
    'p': res5.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.437306  0.525844  0.831626  0.466626
coding                 0.052186  0.334747  0.155896  0.886014
handwritten           -0.132927  0.443578 -0.299669  0.783992
coding:handwritten     0.156098  0.554106  0.281711  0.796480
stats_confidence       0.166862  0.251518  0.663420  0.554487
chebyshev_familiarity -0.110955  0.145033 -0.765027  0.499930
python_skill_level    -0.064398  0.210574 -0.305821  0.779734
Stats_1                0.030494  0.585721  0.052062  0.961752
Stats_2                0.220511  0.734155  0.300361  0.783513
Stats_3                0.084782  0.439229  0.193024  0.859269
Heard_2               -0.147363  0.300629 -0.490182  0.657616
Heard_3                0.031501  0.252667  0.124675  0.908666
Heard_4                0.553168  0.520217  1.063340  0.365632
DSC_minor              0.466865  0.650083  0.718162  0.524534
MATH_major            -0.170907  0.362693 -0.471216  0.669635


## Drop Stats_1

In [26]:
res5 = smf.ols( #Drop Stats_1
    formula='final_score ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res5.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                 -1.174
Method:                 Least Squares   F-statistic:                    0.2801
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.961
Time:                        14:20:20   Log-Likelihood:                 7.0675
No. Observations:                  17   AIC:                             11.87
Df Residuals:                       4   BIC:                             22.70
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.44

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [27]:
coeffs = pd.DataFrame({
    'coef': res5.params,
    'std_err': res5.bse,
    't': res5.tvalues,
    'p': res5.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.448756  0.413837  1.084380  0.339184
coding                 0.042624  0.242482  0.175783  0.869005
handwritten           -0.139630  0.367775 -0.379662  0.723493
coding:handwritten     0.168541  0.433126  0.389127  0.717009
stats_confidence       0.166962  0.217913  0.766188  0.486280
chebyshev_familiarity -0.111302  0.125526 -0.886686  0.425342
python_skill_level    -0.069011  0.165509 -0.416963  0.698111
Stats_2                0.251992  0.360730  0.698563  0.523308
Stats_3                0.093359  0.352772  0.264644  0.804361
Heard_2               -0.144949  0.257353 -0.563230  0.603353
Heard_3                0.034667  0.212480  0.163155  0.878308
Heard_4                0.559038  0.440011  1.270510  0.272772
DSC_minor              0.462834  0.559234  0.827622  0.454403
MATH_major            -0.164520  0.295721 -0.556335  0.607635


## Drop Heard_3

In [28]:
res5 = smf.ols( #Drop Heard_3
    formula='final_score ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_2 + Stats_3 + Heard_2 + Heard_4 + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res5.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                 -1.174
Method:                 Least Squares   F-statistic:                    0.2801
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.961
Time:                        14:20:20   Log-Likelihood:                 7.0675
No. Observations:                  17   AIC:                             11.87
Df Residuals:                       4   BIC:                             22.70
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.48

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [29]:
coeffs = pd.DataFrame({
    'coef': res5.params,
    'std_err': res5.bse,
    't': res5.tvalues,
    'p': res5.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.483423  0.551987  0.875788  0.430588
coding                 0.042624  0.242482  0.175783  0.869005
handwritten           -0.139630  0.367775 -0.379662  0.723493
coding:handwritten     0.168541  0.433126  0.389127  0.717009
stats_confidence       0.166962  0.217913  0.766188  0.486280
chebyshev_familiarity -0.111302  0.125526 -0.886686  0.425342
python_skill_level    -0.069011  0.165509 -0.416963  0.698111
Stats_2                0.251992  0.360730  0.698563  0.523308
Stats_3                0.093359  0.352772  0.264644  0.804361
Heard_2               -0.179616  0.341578 -0.525841  0.626803
Heard_4                0.524371  0.485257  1.080605  0.340674
DSC_minor              0.462834  0.559234  0.827622  0.454403
MATH_major            -0.164520  0.295721 -0.556335  0.607635


## Adjusted Final Score

In [30]:
res6 = smf.ols( #no heard 1 stat, no MATH_minor in the sample
    formula='final_score_adj ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_1 + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res6.summary())

#A negative adjusted R² is a warning. You probably need to:
#Reduce predictors (feature selection)
#Collect more data
#Try a different model

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.633
Model:                            OLS   Adj. R-squared:                 -0.955
Method:                 Least Squares   F-statistic:                    0.3987
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.895
Time:                        14:20:20   Log-Likelihood:                 13.985
No. Observations:                  17   AIC:                           0.02969
Df Residuals:                       3   BIC:                             11.69
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.19

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [31]:
coeffs = pd.DataFrame({
    'coef': res6.params,
    'std_err': res6.bse,
    't': res6.tvalues,
    'p': res6.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.196772  0.298010  0.660288  0.556240
coding                 0.040932  0.222940  0.183599  0.866036
handwritten           -0.090974  0.295421 -0.307946  0.778266
coding:handwritten    -0.002901  0.369032 -0.007862  0.994221
stats_confidence       0.123111  0.167510  0.734947  0.515612
chebyshev_familiarity -0.063143  0.096592 -0.653705  0.559938
python_skill_level    -0.040666  0.140242 -0.289970  0.790726
Stats_1               -0.019835  0.390088 -0.050847  0.962644
Stats_2                0.328228  0.488944  0.671299  0.550096
Stats_3                0.086072  0.292525  0.294237  0.787761
Heard_2               -0.055445  0.192890 -0.287446  0.792483
Heard_3               -0.018128  0.151599 -0.119577  0.912376
Heard_4                0.270346  0.342012  0.790457  0.486984
DSC_major             -0.036834  0.185312 -0.198770  0.855151
DSC_minor              0.233607  0.322162  0.725121  0.520819
MATH_maj

## Drop Stats_1

In [32]:
res6 = smf.ols( #drop stats_1
    formula='final_score_adj ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_2 + Stats_3 + Heard_2 + Heard_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res6.summary())

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.633
Model:                            OLS   Adj. R-squared:                 -0.468
Method:                 Least Squares   F-statistic:                    0.5751
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.794
Time:                        14:20:20   Log-Likelihood:                 13.978
No. Observations:                  17   AIC:                            -1.956
Df Residuals:                       4   BIC:                             8.876
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.19

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [33]:
coeffs = pd.DataFrame({
    'coef': res6.params,
    'std_err': res6.bse,
    't': res6.tvalues,
    'p': res6.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.192071  0.245452  0.782518  0.477643
coding                 0.047151  0.161489  0.291977  0.784822
handwritten           -0.086613  0.244932 -0.353623  0.741473
coding:handwritten    -0.010995  0.288455 -0.038117  0.971421
stats_confidence       0.123046  0.145126  0.847854  0.444273
chebyshev_familiarity -0.062916  0.083598 -0.752604  0.493555
python_skill_level    -0.037666  0.110226 -0.341712  0.749765
Stats_2                0.307751  0.240240  1.281016  0.269416
Stats_3                0.080493  0.234940  0.342611  0.749138
Heard_2               -0.056100  0.166747 -0.336441  0.753448
Heard_3               -0.019272  0.129891 -0.148368  0.889231
Heard_4                0.267443  0.292161  0.915395  0.411776
DSC_major             -0.040496  0.147935 -0.273742  0.797837
DSC_minor              0.232567  0.278558  0.834896  0.450740
MATH_major            -0.049265  0.196945 -0.250144  0.814798


## Drop Heard_3

In [34]:
res6 = smf.ols( #drop Heard_3
    formula='final_score_adj ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_2 + Stats_3 + Heard_2 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res6.summary())

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.633
Model:                            OLS   Adj. R-squared:                 -0.468
Method:                 Least Squares   F-statistic:                    0.5751
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.794
Time:                        14:20:20   Log-Likelihood:                 13.978
No. Observations:                  17   AIC:                            -1.956
Df Residuals:                       4   BIC:                             8.876
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.17

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [35]:
coeffs = pd.DataFrame({
    'coef': res6.params,
    'std_err': res6.bse,
    't': res6.tvalues,
    'p': res6.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.179223  0.289917  0.618187  0.569913
coding                 0.047151  0.161489  0.291977  0.784822
handwritten           -0.086613  0.244932 -0.353623  0.741473
coding:handwritten    -0.010995  0.288455 -0.038117  0.971421
stats_confidence       0.123046  0.145126  0.847854  0.444273
chebyshev_familiarity -0.062916  0.083598 -0.752604  0.493555
python_skill_level    -0.037666  0.110226 -0.341712  0.749765
Stats_2                0.307751  0.240240  1.281016  0.269416
Stats_3                0.080493  0.234940  0.342611  0.749138
Heard_2               -0.036829  0.227485 -0.161894  0.879238
Heard_4                0.286715  0.323173  0.887187  0.425102
DSC_major             -0.046920  0.161677 -0.290208  0.786080
DSC_minor              0.226143  0.291963  0.774560  0.481837
MATH_major            -0.049265  0.196945 -0.250144  0.814798


## Drop Heard_2

In [36]:
res6 = smf.ols( #drop Heard_2
    formula='final_score_adj ~ coding*handwritten + stats_confidence + chebyshev_familiarity + python_skill_level + Stats_2 + Stats_3 + Heard_4 + DSC_major + DSC_minor + MATH_major',
    data=df_encoded
).fit()

print(res6.summary())

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.631
Model:                            OLS   Adj. R-squared:                 -0.182
Method:                 Least Squares   F-statistic:                    0.7761
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.664
Time:                        14:20:20   Log-Likelihood:                 13.922
No. Observations:                  17   AIC:                            -3.845
Df Residuals:                       5   BIC:                             6.154
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.15

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=17 observations were given.
  return hypotest_fun_in(*args, **kwds)


In [37]:
coeffs = pd.DataFrame({
    'coef': res6.params,
    'std_err': res6.bse,
    't': res6.tvalues,
    'p': res6.pvalues
})

print(coeffs)

                           coef   std_err         t         p
Intercept              0.156578  0.227876  0.687119  0.522573
coding                 0.053902  0.139997  0.385024  0.716054
handwritten           -0.073983  0.208343 -0.355103  0.737000
coding:handwritten    -0.030749  0.234548 -0.131099  0.900809
stats_confidence       0.119860  0.129027  0.928955  0.395548
chebyshev_familiarity -0.059442  0.072503 -0.819854  0.449600
python_skill_level    -0.035362  0.098084 -0.360531  0.733181
Stats_2                0.322386  0.199736  1.614056  0.167435
Stats_3                0.093741  0.197621  0.474350  0.655243
Heard_4                0.280496  0.287944  0.974133  0.374730
DSC_major             -0.054013  0.139653 -0.386764  0.714844
DSC_minor              0.210591  0.247406  0.851194  0.433524
MATH_major            -0.043550  0.173867 -0.250477  0.812185


## Adjusted Final score DROPPED: Stats_1, Heard_3, Heard_2
## Final score DROPPED: DSC_major, Stats_1, Heard_3